# Objective 2 — Dashboard Preparation
**AI Trust Paradox Among Software Developers**
AICW Fellowship 2026 | Edunet Foundation | Galgotias University

### Running in Jupyter (Local)
1. Make sure `trust_paradox_clean.csv` (output of Objective 1) is in the **same folder**
2. Run **Cell 1** (imports)
3. Run **Cell 2** to set path (edit if needed)
4. Run all remaining cells in order
5. Output CSVs will be saved in the same folder




In [2]:
# CELL 1 — Install and import libraries (JupyterLite / Pyodide)

import sys, micropip
await micropip.install('pandas')
await micropip.install('numpy')
await micropip.install('scipy')
await micropip.install('ipywidgets')

import pandas as pd
import numpy  as np
import scipy                    # ← top-level scipy for version
from scipy import stats
import warnings, io, os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
pd.set_option('display.max_columns', 50)

print('Libraries loaded:')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  scipy   : {scipy.__version__}')   # ← scipy.__version__ not stats.__version__
print()
print('✓ All libraries ready')
print('  Run Cell 2 to upload trust_paradox_clean.csv')

Libraries loaded:
  pandas  : 2.3.2
  numpy   : 2.2.5
  scipy   : 1.14.1

✓ All libraries ready
  Run Cell 2 to upload trust_paradox_clean.csv


In [3]:
# CELL 2 — Upload trust_paradox_clean.csv  (output from Objective 1)

import ipywidgets as widgets
from IPython.display import display

upload_widget = widgets.FileUpload(
    accept      = '.csv',
    multiple    = False,
    description = 'Upload CSV',
    layout      = widgets.Layout(width='260px')
)

display(upload_widget)

print()
print('Click "Upload" → select trust_paradox_clean.csv')
print('This is the cleaned file produced by Objective 1.')
print('File size ~128 MB — wait until filename appears next to button.')
print('When done → run Cell 3')

FileUpload(value=(), accept='.csv', description='Upload CSV', layout=Layout(width='260px'))


Click "Upload" → select trust_paradox_clean.csv
This is the cleaned file produced by Objective 1.
File size ~128 MB — wait until filename appears next to button.
When done → run Cell 3


## Cell 3 — Load and verify the cleaned dataset

In [4]:
# CELL 3 — Load and verify the cleaned dataset

import io, pandas as pd

if len(upload_widget.value) == 0:
    print('NO FILE UPLOADED — go back to Cell 2 and upload first.')
else:
    # ipywidgets v8+: .value is a tuple not a dict
    uploaded_file = upload_widget.value[0]
    filename      = uploaded_file['name']
    file_bytes    = bytes(uploaded_file['content'])
    size_mb       = len(file_bytes) / 1024 / 1024

    print(f'Uploaded : {filename}  ({size_mb:.0f} MB)')
    print('Loading...')

    df = pd.read_csv(io.BytesIO(file_bytes), low_memory=False)

    print(f'Rows     : {len(df):,}')
    print(f'Columns  : {len(df.columns)}')

    # Verify correct file
    assert len(df) == 48195, f'Expected 48,195 rows — got {len(df):,}. Did you upload the CLEANED file?'

    REQUIRED = [
        'AIAcc_clean','AIAcc_score','AISent_clean','AISent_score',
        'AIThreat_clean','JobSat_clean','UsesAI','ExperienceBand',
        'ToolGroup','YearsCode_num','Frust_AlmostRight',
    ]
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        print(f'MISSING COLUMNS: {missing}')
        print('Make sure you uploaded trust_paradox_clean.csv from Objective 1')
    else:
        print()
        print('✓ All required columns present')
        print('✓ Dataset verified: 48,195 rows')
        print('  Run Cell 4 onwards')

Uploaded : trust_paradox_clean Jupter Final.csv  (128 MB)
Loading...
Rows     : 48,195
Columns  : 187

✓ All required columns present
✓ Dataset verified: 48,195 rows
  Run Cell 4 onwards


---
# PART A — Paradox 1: Use-Despite-Distrust
**Variables used:** AIAcc_clean, AIAcc_score, UsesAI, ExperienceBand, Country, DevType

## Cell 4 — P1: Trust level distribution (headline KPI)

In [5]:
# Trust level distribution — excludes No Response from denominator
trust_ans = df[df['AIAcc_clean'] != 'No Response'].copy()
n_trust   = len(trust_ans)

# Adoption rate
uses_ans   = df[df['UsesAI'] != 'No Response']
adopt_rate = (uses_ans['UsesAI'] == 'Uses AI').mean() * 100

# Trust distribution
TRUST_ORDER = [
    'Highly trust', 'Somewhat trust',
    'Neither trust nor distrust',
    'Somewhat distrust', 'Highly distrust'
]

print('=' * 55)
print('  PARADOX 1 — KEY PERFORMANCE INDICATORS')
print('=' * 55)
print(f'\n  Adoption rate        : {adopt_rate:.1f}%  (target: 78.5%)')
print(f'  Answered trust Q     : {n_trust:,}')
print(f'  No Response (trust Q): {(df["AIAcc_clean"]=="No Response").sum():,}  (MNAR — retained)\n')

print(f'  {"Trust Level":<35} {"Count":>7} {"Percent":>8}')
print(f'  {"-"*52}')

p1_dist = []
for level in TRUST_ORDER:
    cnt = int((trust_ans['AIAcc_clean'] == level).sum())
    pct = cnt / n_trust * 100
    bar = '█' * int(pct / 2)
    print(f'  {level:<35} {cnt:>7,}  {pct:>6.1f}%  {bar}')
    p1_dist.append({'Trust_Level': level, 'Count': cnt, 'Percent': round(pct, 1)})

# Aggregate metrics
ht_pct  = (trust_ans['AIAcc_clean'] == 'Highly trust').mean()  * 100
ad_pct  = trust_ans['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
mean_sc = df['AIAcc_score'].mean()

print(f'\n  HEADLINE NUMBERS FOR POWER BI CARDS:')
print(f'  Adoption Rate         = {adopt_rate:.1f}%')
print(f'  High Trust Rate       = {ht_pct:.1f}%')
print(f'  Any Distrust Rate     = {ad_pct:.1f}%')
print(f'  Avg Trust Score       = {mean_sc:.3f}  (below 3.0 = distrust direction)')
print(f'  Paradox Severity      = {adopt_rate - ht_pct:.1f} pts')

# Save for export
p1_dist_df = pd.DataFrame(p1_dist)
print('\n[P1 distribution saved for export]')

  PARADOX 1 — KEY PERFORMANCE INDICATORS

  Adoption rate        : 78.5%  (target: 78.5%)
  Answered trust Q     : 32,717
  No Response (trust Q): 15,478  (MNAR — retained)

  Trust Level                           Count  Percent
  ----------------------------------------------------
  Highly trust                          1,000     3.1%  █
  Somewhat trust                        9,668    29.6%  ██████████████
  Neither trust nor distrust            7,039    21.5%  ██████████
  Somewhat distrust                     8,549    26.1%  █████████████
  Highly distrust                       6,461    19.7%  █████████

  HEADLINE NUMBERS FOR POWER BI CARDS:
  Adoption Rate         = 78.5%
  High Trust Rate       = 3.1%
  Any Distrust Rate     = 45.9%
  Avg Trust Score       = 2.700  (below 3.0 = distrust direction)
  Paradox Severity      = 75.5 pts

[P1 distribution saved for export]


## Cell 5 — P1: Cross-tabulation UsesAI × AIAcc_clean (the core Paradox 1 chart)

In [6]:
# This is the stacked bar chart in Power BI Page 2
# Shows: among active AI users, what % distrust vs trust?

# Filter: only rows where BOTH UsesAI and AIAcc_clean are answered
both_ans = df[
    (df['UsesAI'] != 'No Response') &
    (df['AIAcc_clean'] != 'No Response')
].copy()

print('Cross-tabulation: UsesAI × AIAcc_clean')
print('(% of row — each row sums to 100%)')
print()

crosstab_rows = []
for group in ['Uses AI', 'Does Not Use AI']:
    grp = both_ans[both_ans['UsesAI'] == group]
    n   = len(grp)
    row = {'UsesAI_Group': group, 'n': n}
    print(f'{group} (n={n:,}):')
    for level in TRUST_ORDER:
        cnt = int((grp['AIAcc_clean'] == level).sum())
        pct = cnt / n * 100
        row[level] = round(pct, 1)
        bar = '█' * int(pct / 3)
        print(f'  {level:<35} {pct:>5.1f}%  {bar}')
    any_dist = grp['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    row['Any_Distrust_Pct'] = round(any_dist, 1)
    print(f'  → ANY DISTRUST: {any_dist:.1f}%')
    print()
    crosstab_rows.append(row)

# Chi-square test
ct = pd.crosstab(both_ans['UsesAI'], both_ans['AIAcc_clean'])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)
n_ct   = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n_ct * (min(ct.shape) - 1)))

print(f'Chi-square test of independence:')
print(f'  χ² = {chi2:,.2f}')
print(f'  p  = {p_val:.4f}  ({"SIGNIFICANT p<0.0001" if p_val < 0.0001 else "not significant"})')
print(f'  df = {dof}')
print(f'  Cramér\'s V = {cramers_v:.3f}  (>0.3 = LARGE effect)')

p1_crosstab_df = pd.DataFrame(crosstab_rows)
print('\n[P1 cross-tab saved for export]')

Cross-tabulation: UsesAI × AIAcc_clean
(% of row — each row sums to 100%)

Uses AI (n=25,678):
  Highly trust                          3.7%  █
  Somewhat trust                       35.5%  ███████████
  Neither trust nor distrust           23.4%  ███████
  Somewhat distrust                    25.7%  ████████
  Highly distrust                      11.7%  ███
  → ANY DISTRUST: 37.4%

Does Not Use AI (n=7,005):
  Highly trust                          0.8%  
  Somewhat trust                        7.8%  ██
  Neither trust nor distrust           14.5%  ████
  Somewhat distrust                    27.6%  █████████
  Highly distrust                      49.3%  ████████████████
  → ANY DISTRUST: 76.9%

Chi-square test of independence:
  χ² = 5,728.86
  p  = 0.0000  (SIGNIFICANT p<0.0001)
  df = 4
  Cramér's V = 0.419  (>0.3 = LARGE effect)

[P1 cross-tab saved for export]


## Cell 6 — P1: Expertise effect (trust by ExperienceBand)

In [7]:
# The most counterintuitive finding: trust FALLS as experience RISES

print('Trust by ExperienceBand — the expertise effect')
print()
print(f'{"Band":<30} {"n":>7} {"High Trust %":>13} {"Any Distrust %":>15}')
print('-' * 68)

BAND_ORDER = [
    'Early Career (0-2 yrs)',
    'Mid-Early (3-5 yrs)',
    'Mid (6-10 yrs)',
    'Experienced (10+ yrs)'
]

exp_rows = []
for band in BAND_ORDER:
    g   = trust_ans[trust_ans['ExperienceBand'] == band]
    n   = len(g)
    ht  = (g['AIAcc_clean'] == 'Highly trust').mean()  * 100
    ad  = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    avg = g['AIAcc_score'].mean()
    print(f'{band:<30} {n:>7,} {ht:>12.1f}% {ad:>14.1f}%')
    exp_rows.append({
        'ExperienceBand': band, 'n': n,
        'High_Trust_Pct': round(ht,1),
        'Any_Distrust_Pct': round(ad,1),
        'Avg_Trust_Score': round(avg,3)
    })

print()
print('Key insight: Early Career 10.0% trust → Experienced 2.3% trust')
print('Trust FALLS monotonically at every experience level.')
print('Experienced developers also show HIGHEST distrust (49.8%)')

p1_expertise_df = pd.DataFrame(exp_rows)
print('\n[P1 expertise table saved for export]')

Trust by ExperienceBand — the expertise effect

Band                                 n  High Trust %  Any Distrust %
--------------------------------------------------------------------
Early Career (0-2 yrs)             941         10.0%           22.8%
Mid-Early (3-5 yrs)              3,366          5.4%           34.9%
Mid (6-10 yrs)                   7,618          2.8%           43.8%
Experienced (10+ yrs)           20,442          2.3%           49.8%

Key insight: Early Career 10.0% trust → Experienced 2.3% trust
Trust FALLS monotonically at every experience level.
Experienced developers also show HIGHEST distrust (49.8%)

[P1 expertise table saved for export]


## Cell 7 — P1: Country trust analysis

In [8]:
# Country-level trust analysis
# Only countries with at least 200 respondents who answered the trust question

country_rows = []
country_data = trust_ans.groupby('Country').apply(
    lambda g: pd.Series({
        'n': len(g),
        'High_Trust_Pct': round((g['AIAcc_clean']=='Highly trust').mean()*100, 1),
        'Any_Distrust_Pct': round(g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean()*100, 1),
        'Avg_Score': round(g['AIAcc_score'].mean(), 3)
    })
).reset_index()

# Only countries with n >= 200
country_data = country_data[country_data['n'] >= 200].sort_values('High_Trust_Pct', ascending=False)

print(f'Countries with 200+ respondents: {len(country_data)}')
print()
print(f'{"Country":<45} {"n":>6} {"High Trust":>11} {"Any Distrust":>13}')
print('-' * 78)
for _, row in country_data.head(15).iterrows():
    print(f'{str(row["Country"])[:44]:<45} {int(row["n"]):>6,} {row["High_Trust_Pct"]:>10.1f}% {row["Any_Distrust_Pct"]:>12.1f}%')

print()
top    = country_data.iloc[0]
bottom = country_data.iloc[-1]
print(f'Highest trust: {top["Country"]} ({top["High_Trust_Pct"]}%)')
print(f'Lowest trust:  {bottom["Country"]} ({bottom["High_Trust_Pct"]}%)')

p1_country_df = country_data.copy()
print('\n[P1 country data saved for export]')

Countries with 200+ respondents: 35

Country                                            n  High Trust  Any Distrust
------------------------------------------------------------------------------
Pakistan                                         200       11.0%         16.5%
China                                            230        9.6%         23.5%
India                                          2,129        7.5%         16.9%
Mexico                                           263        6.5%         25.5%
Turkey                                           263        5.7%         26.2%
Brazil                                           766        5.1%         37.6%
South Africa                                     238        4.2%         41.2%
Argentina                                        206        3.4%         36.4%
Bulgaria                                         231        3.0%         38.5%
Ukraine                                          864        2.5%         28.0%
Netherlands    

---
# PART B — Paradox 2: Adoption-Ethics Gap
**Variables used:** AISent_clean, AISent_score, AIAcc_clean, ToolGroup, Frust_ columns

## Cell 8 — P2: Ethics gap (sentiment vs trust)

In [9]:
# The ethics gap = positive sentiment rate MINUS positive trust rate

sent_ans = df[df['AISent_clean'].isin([
    'Very favorable','Favorable','Indifferent','Unfavorable','Very unfavorable'
])].copy()

pos_sent  = sent_ans['AISent_clean'].isin(['Very favorable','Favorable']).mean() * 100
pos_trust = trust_ans['AIAcc_clean'].isin(['Highly trust','Somewhat trust']).mean()  * 100
gap       = pos_sent - pos_trust

mean_sent  = df['AISent_score'].mean()
mean_trust = df['AIAcc_score'].mean()

print('=' * 55)
print('  PARADOX 2 — ETHICS GAP ANALYSIS')
print('=' * 55)
print(f'\n  Positive Sentiment (Very/Favorable) : {pos_sent:.1f}%  (n={sent_ans["AISent_clean"].isin(["Very favorable","Favorable"]).sum():,})')
print(f'  Positive Trust  (Highly/Somewhat)   : {pos_trust:.1f}%  (n={trust_ans["AIAcc_clean"].isin(["Highly trust","Somewhat trust"]).sum():,})')
print(f'  ETHICS GAP                          : {gap:.1f} percentage points')
print(f'\n  Mean AISent_score : {mean_sent:.3f}  (above 3.0 = positive direction)')
print(f'  Mean AIAcc_score  : {mean_trust:.3f}  (below 3.0 = distrust direction)')
print(f'  Score gap         : {mean_sent - mean_trust:.3f} pts on 1-5 scale')

print(f'\nSentiment distribution (n={len(sent_ans):,}):')
SENT_ORDER = ['Very favorable','Favorable','Indifferent','Unfavorable','Very unfavorable']
p2_sent_rows = []
for level in SENT_ORDER:
    cnt = int((sent_ans['AISent_clean']==level).sum())
    pct = cnt/len(sent_ans)*100
    bar = '█' * int(pct/2)
    print(f'  {level:<20} {cnt:>7,}  {pct:>5.1f}%  {bar}')
    p2_sent_rows.append({'Sentiment_Level':level,'Count':cnt,'Percent':round(pct,1)})

# Ethics gap summary
p2_gap_df = pd.DataFrame([
    {'Dimension':'Positive Sentiment','Rate':round(pos_sent,1),'n':len(sent_ans),'Mean_Score':round(mean_sent,3)},
    {'Dimension':'Positive Trust',    'Rate':round(pos_trust,1),'n':len(trust_ans),'Mean_Score':round(mean_trust,3)},
    {'Dimension':'Ethics Gap (pp)',   'Rate':round(gap,1),'n':0,'Mean_Score':round(mean_sent-mean_trust,3)}
])
p2_sent_df = pd.DataFrame(p2_sent_rows)
print('\n[P2 ethics gap and sentiment distribution saved for export]')

  PARADOX 2 — ETHICS GAP ANALYSIS

  Positive Sentiment (Very/Favorable) : 61.0%  (n=19,609)
  Positive Trust  (Highly/Somewhat)   : 32.6%  (n=10,668)
  ETHICS GAP                          : 28.4 percentage points

  Mean AISent_score : 3.535  (above 3.0 = positive direction)
  Mean AIAcc_score  : 2.700  (below 3.0 = distrust direction)
  Score gap         : 0.834 pts on 1-5 scale

Sentiment distribution (n=32,142):
  Very favorable         7,504   23.3%  ███████████
  Favorable             12,105   37.7%  ██████████████████
  Indifferent            5,783   18.0%  ████████
  Unfavorable            3,571   11.1%  █████
  Very unfavorable       3,179    9.9%  ████

[P2 ethics gap and sentiment distribution saved for export]


## Cell 9 — P2: ToolGroup distrust analysis (Paradox 2 Proof B)

In [10]:
# Familiarity breeds scepticism: more tools = MORE distrust

# Only rows where BOTH ToolGroup and AIAcc_clean are answered
tool_trust = df[
    (df['ToolGroup'] != 'No Response') &
    (df['AIAcc_clean'] != 'No Response')
].copy()

print('Any Distrust Rate by ToolGroup:')
print('(Expected: 41.4% → 41.4% → 47.5%)')
print()

TOOL_ORDER = ['1 Tool', '2-3 Tools', '4+ Tools']
p2_tool_rows = []
for grp in TOOL_ORDER:
    g   = tool_trust[tool_trust['ToolGroup'] == grp]
    n   = len(g)
    ad  = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    ht  = (g['AIAcc_clean'] == 'Highly trust').mean() * 100
    avg = g['AIAcc_score'].mean()
    bar = '█' * int(ad / 2)
    print(f'  {grp:<12}  n={n:>6,}  Any Distrust={ad:>5.1f}%  High Trust={ht:>4.1f}%  {bar}')
    p2_tool_rows.append({
        'ToolGroup': grp, 'n': n,
        'Any_Distrust_Pct': round(ad,1),
        'High_Trust_Pct': round(ht,1),
        'Avg_Trust_Score': round(avg,3)
    })

print()
distrust_vals = [r['Any_Distrust_Pct'] for r in p2_tool_rows]
print(f'Direction: {distrust_vals[0]} → {distrust_vals[1]} → {distrust_vals[2]}')
print(f'Rise from 1-Tool to 4+Tools: {distrust_vals[2]-distrust_vals[0]:.1f} pp')
print('Conclusion: MORE tools used = MORE distrust. Familiarity breeds scepticism.')

p2_tool_df = pd.DataFrame(p2_tool_rows)
print('\n[P2 ToolGroup distrust table saved for export]')

Any Distrust Rate by ToolGroup:
(Expected: 41.4% → 41.4% → 47.5%)

  1 Tool        n=   867  Any Distrust= 41.4%  High Trust= 4.5%  ████████████████████
  2-3 Tools     n= 4,121  Any Distrust= 41.4%  High Trust= 3.3%  ████████████████████
  4+ Tools      n=20,951  Any Distrust= 47.5%  High Trust= 2.6%  ███████████████████████

Direction: 41.4 → 41.4 → 47.5
Rise from 1-Tool to 4+Tools: 6.1 pp
Conclusion: MORE tools used = MORE distrust. Familiarity breeds scepticism.

[P2 ToolGroup distrust table saved for export]


## Cell 10 — P2: AIFrustration breakdown

In [11]:
# Frustration counts — explains WHY distrust grows with exposure

frust_cols = {
    'Frust_AlmostRight'      : '"Almost right, needs changes every time"',
    'Frust_Debugging'        : '"Debugging AI code is more time-consuming"',
    'Frust_LessConfident'    : '"Makes me less confident in own skills"',
    'Frust_HardToUnderstand' : '"Hard to understand why code worked"',
}

print('AIFrustration breakdown (multi-select — overlaps allowed):')
print(f'  Total respondents       : {len(df):,}')
print(f'  Answered AIFrustration  : {df["AIFrustration"].notna().sum():,}  ({df["AIFrustration"].notna().mean()*100:.1f}%)')
print()
print(f'  {"Frustration":<48} {"Count":>8} {"% of total":>11}')
print('  ' + '-'*70)

p2_frust_rows = []
for col, label in frust_cols.items():
    if col in df.columns:
        cnt = int(df[col].sum())
        pct = cnt / len(df) * 100
        bar = '█' * int(pct / 2)
        print(f'  {label:<48} {cnt:>8,} {pct:>10.1f}%  {bar}')
        p2_frust_rows.append({'Frustration': label, 'Count': cnt, 'Pct_of_Total': round(pct,1)})

print()
print('The top frustration "Almost right, needs changes" explains the ethics gap:')
print('Developers enjoy the speed boost (positive sentiment) but spend extra time')
print('fixing outputs — which builds distrust rather than trust over time.')

p2_frust_df = pd.DataFrame(p2_frust_rows)
print('\n[P2 frustration data saved for export]')

AIFrustration breakdown (multi-select — overlaps allowed):
  Total respondents       : 48,195
  Answered AIFrustration  : 30,985  (64.3%)

  Frustration                                         Count  % of total
  ----------------------------------------------------------------------
  "Almost right, needs changes every time"           20,465       42.5%  █████████████████████
  "Debugging AI code is more time-consuming"         14,041       29.1%  ██████████████
  "Makes me less confident in own skills"             6,207       12.9%  ██████
  "Hard to understand why code worked"                5,050       10.5%  █████

The top frustration "Almost right, needs changes" explains the ethics gap:
Developers enjoy the speed boost (positive sentiment) but spend extra time
fixing outputs — which builds distrust rather than trust over time.

[P2 frustration data saved for export]


---
# PART C — Paradox 3: Utility-Threat Tension
**Variables used:** AIThreat_clean, NewRole, JobSat, UsesAI, ExperienceBand

## Cell 11 — P3: Job threat distribution (current state)

In [12]:
# Core Paradox 3 chart — donut distribution

threat_ans = df[df['AIThreat_clean'] != 'No Response'].copy()
n_threat   = len(threat_ans)

print('=' * 55)
print('  PARADOX 3 — JOB THREAT ANALYSIS')
print('=' * 55)
print(f'\nRespondents who answered AIThreat: {n_threat:,}')
print(f'No Response (skipped):             {(df["AIThreat_clean"]=="No Response").sum():,}')
print()

THREAT_ORDER = ['No', "I'm not sure", 'Yes']
p3_threat_rows = []
for val in THREAT_ORDER:
    cnt = int((threat_ans['AIThreat_clean'] == val).sum())
    pct = cnt / n_threat * 100
    bar = '█' * int(pct / 2)
    label = {'No':'Not Threatened',"I'm not sure":'Uncertain','Yes':'Threatened'}[val]
    print(f'  {label:<18} {cnt:>7,}  {pct:>5.1f}%  {bar}')
    p3_threat_rows.append({'Response':val,'Label':label,'Count':cnt,'Percent':round(pct,1)})

not_conf = threat_ans['AIThreat_clean'].isin(["I'm not sure",'Yes']).sum()
not_conf_pct = not_conf / n_threat * 100

print(f'\n  Not confident job is safe  = {not_conf:,}  ({not_conf_pct:.1f}%) [Yes + Not Sure]')
print(f'  Paradox 3 headline         : 36.2% are NOT confident their job is safe')
print(f'  Declining trend            : 70% (2023) → 68% (2024) → 63.8% (2025)')

p3_threat_df = pd.DataFrame(p3_threat_rows)
print('\n[P3 threat distribution saved for export]')

  PARADOX 3 — JOB THREAT ANALYSIS

Respondents who answered AIThreat: 35,402
No Response (skipped):             12,793

  Not Threatened      22,576   63.8%  ███████████████████████████████
  Uncertain            7,553   21.3%  ██████████
  Threatened           5,273   14.9%  ███████

  Not confident job is safe  = 12,826  (36.2%) [Yes + Not Sure]
  Paradox 3 headline         : 36.2% are NOT confident their job is safe
  Declining trend            : 70% (2023) → 68% (2024) → 63.8% (2025)

[P3 threat distribution saved for export]


## Cell 12 — P3: 3-year trend data (manually entered — used in Power BI)

In [13]:
# 2023 and 2024 figures are from Stack Overflow published reports
# 2025 figures are from your dataset (calculated above)

trend_data = {
    'Year': [2023, 2024, 2025],
    'Not_Threatened': [0.70, 0.68, 0.638],
    'Threatened':     [0.12, 0.13, 0.149],
    'Not_Sure':       [0.18, 0.19, 0.213]
}
p3_trend_df = pd.DataFrame(trend_data)

print('3-year trend data (ready for Power BI Enter Data):')
print()
print(f'{"Year":<8} {"Not Threatened":>15} {"Threatened":>12} {"Not Sure":>10}')
print('-' * 48)
for _, row in p3_trend_df.iterrows():
    nt_bar = '█' * int(row['Not_Threatened'] * 20)
    t_bar  = '█' * int(row['Threatened']     * 20)
    print(f'{int(row["Year"]):<8} {row["Not_Threatened"]*100:>14.1f}%  {row["Threatened"]*100:>11.1f}%  {row["Not_Sure"]*100:>9.1f}%')

decline = (trend_data['Not_Threatened'][0] - trend_data['Not_Threatened'][2]) * 100
rise    = (trend_data['Threatened'][2]     - trend_data['Threatened'][0])     * 100
print(f'\nNot Threatened fell : {decline:.1f} pp over 2 years')
print(f'Threatened rose     : {rise:.1f} pp over 2 years')

# Statistical test (2025 data vs historical expectation)
# Test independence of year and threat response
print('\nChi-square test (2025 threat distribution vs historical baseline):')
observed = [22576, 7553, 5273]  # No, Not Sure, Yes
expected_pcts = [0.70, 0.18, 0.12]  # 2023 baseline
total = sum(observed)
expected_counts = [p * total for p in expected_pcts]
chi2, p_val = stats.chisquare(observed, f_exp=expected_counts)
print(f'  χ² = {chi2:,.2f}')
print(f'  p  = {p_val:.6f}  (p < 0.0001 — highly significant)')
print('  Conclusion: decline from 70% to 63.8% is statistically significant')

print('\n[P3 trend data saved for export]')

3-year trend data (ready for Power BI Enter Data):

Year      Not Threatened   Threatened   Not Sure
------------------------------------------------
2023               70.0%         12.0%       18.0%
2024               68.0%         13.0%       19.0%
2025               63.8%         14.9%       21.3%

Not Threatened fell : 6.2 pp over 2 years
Threatened rose     : 2.9 pp over 2 years

Chi-square test (2025 threat distribution vs historical baseline):
  χ² = 662.20
  p  = 0.000000  (p < 0.0001 — highly significant)
  Conclusion: decline from 70% to 63.8% is statistically significant

[P3 trend data saved for export]


## Cell 13 — P3: Threat by ExperienceBand (opposite of Paradox 1 expertise effect)

In [14]:
# In P1: trust FALLS with experience (experienced most sceptical)
# In P3: threat FALLS with experience (experienced least threatened)
# This cross-paradox comparison is one of the strongest insights

print('Threat rate by ExperienceBand:')
print('(Compare with P1 trust rates — they move in OPPOSITE directions)')
print()
print(f'{"Band":<30} {"n":>7} {"Threatened %":>14} {"P1 High Trust %":>17}')
print('-' * 72)

p1_trust_ref = {
    'Early Career (0-2 yrs)': 10.0,
    'Mid-Early (3-5 yrs)': 5.4,
    'Mid (6-10 yrs)': 2.8,
    'Experienced (10+ yrs)': 2.3
}

p3_exp_rows = []
for band in BAND_ORDER:
    g   = threat_ans[threat_ans['ExperienceBand'] == band]
    n   = len(g)
    thr = (g['AIThreat_clean'] == 'Yes').mean() * 100
    ht  = p1_trust_ref.get(band, 0)
    arrow = '← MOST THREATENED' if band == 'Early Career (0-2 yrs)' else '← LEAST THREATENED' if band == 'Experienced (10+ yrs)' else ''
    print(f'{band:<30} {n:>7,} {thr:>13.1f}% {ht:>16.1f}%  {arrow}')
    p3_exp_rows.append({
        'ExperienceBand': band, 'n': n,
        'Threatened_Pct': round(thr,1),
        'P1_HighTrust_Pct': ht
    })

print()
print('KEY INSIGHT: Paradox 1 and Paradox 3 move in OPPOSITE directions:')
print('  Experienced developers: LOWEST trust (2.3%) but LEAST threatened (13.6%)')
print('  Early Career developers: HIGHEST trust (10.0%) but MOST threatened (21.2%)')
print()
print('Explanation: Experienced devs know AI cannot easily replace deep expertise.')
print('Junior devs trust AI more (less experience to detect errors) but fear')
print('displacement more (their skills are more easily replicable by AI).')

p3_exp_df = pd.DataFrame(p3_exp_rows)
print('\n[P3 threat by experience saved for export]')

Threat rate by ExperienceBand:
(Compare with P1 trust rates — they move in OPPOSITE directions)

Band                                 n   Threatened %   P1 High Trust %
------------------------------------------------------------------------
Early Career (0-2 yrs)           1,225          21.2%             10.0%  ← MOST THREATENED
Mid-Early (3-5 yrs)              3,912          17.7%              5.4%  
Mid (6-10 yrs)                   8,353          14.6%              2.8%  
Experienced (10+ yrs)           21,295          13.6%              2.3%  ← LEAST THREATENED

KEY INSIGHT: Paradox 1 and Paradox 3 move in OPPOSITE directions:
  Experienced developers: LOWEST trust (2.3%) but LEAST threatened (13.6%)
  Early Career developers: HIGHEST trust (10.0%) but MOST threatened (21.2%)

Explanation: Experienced devs know AI cannot easily replace deep expertise.
Junior devs trust AI more (less experience to detect errors) but fear
displacement more (their skills are more easily replicable by

## Cell 14 — P3: NewRole career disruption analysis

In [15]:
# NewRole: have you considered or made a career change because of AI?

nr_ans = df[df['NewRole'].notna()].copy()
n_nr   = len(nr_ans)

print(f'Career change analysis (n={n_nr:,} who answered NewRole):')
print()

p3_nr_rows = []
for val, cnt in nr_ans['NewRole'].value_counts().items():
    pct = cnt / n_nr * 100
    short = str(val)[:60]
    action = 'STABLE' if 'neither' in str(val).lower() else \
             'CONCERN' if 'somewhat' in str(val).lower() else \
             'HIGH CONCERN' if 'strongly' in str(val).lower() else \
             'DISRUPTED' if 'voluntary' in str(val).lower() else 'FORCED'
    bar = '█' * int(pct / 2)
    print(f'  [{action:<12}] {cnt:>6,}  {pct:>5.1f}%  {bar}')
    print(f'    "{short}"')
    p3_nr_rows.append({'NewRole': str(val)[:80], 'Count': cnt, 'Percent': round(pct,1), 'Category': action})

# Calculate: at least considered
at_least = nr_ans[~nr_ans['NewRole'].str.lower().str.contains('neither', na=False)]
at_pct   = len(at_least) / n_nr * 100

print(f'\n  HEADLINE: {at_pct:.1f}% have at least considered a career change')
print(f'  Of these, {len(nr_ans[nr_ans["NewRole"].str.contains("involuntarily", na=False)]):,} transitioned INVOLUNTARILY')

p3_nr_df = pd.DataFrame(p3_nr_rows)
print('\n[P3 NewRole data saved for export]')

Career change analysis (n=34,867 who answered NewRole):

  [STABLE      ] 15,959   45.8%  ██████████████████████
    "I have neither consider or transitioned into a new career or"
  [CONCERN     ] 10,056   28.8%  ██████████████
    "I have somewhat considered changing my career and/or the ind"
  [HIGH CONCERN]  5,113   14.7%  ███████
    "I have strongly considered changing my career and/or the ind"
  [FORCED      ]  3,039    8.7%  ████
    "I have transitioned into a new career and/or industry volunt"
  [FORCED      ]    700    2.0%  █
    "I have transitioned into a new career and/or industry involu"

  HEADLINE: 54.2% have at least considered a career change
  Of these, 700 transitioned INVOLUNTARILY

[P3 NewRole data saved for export]


## Cell 15 — P3: JobSat comparison (AI users vs non-users)

In [16]:
# Job satisfaction by AI adoption — supporting evidence for Paradox 3

jobsat_data = df[df['JobSat'].notna()].copy()

ai_grp  = jobsat_data[jobsat_data['UsesAI'] == 'Uses AI']['JobSat']
non_grp = jobsat_data[jobsat_data['UsesAI'] == 'Does Not Use AI']['JobSat']

print('Job Satisfaction by AI Adoption Status:')
print(f'  (Only employed respondents who answered JobSat: n={len(jobsat_data):,})')
print()
print(f'  AI users       : n={len(ai_grp):,}  mean={ai_grp.mean():.2f}  median={ai_grp.median():.1f}')
print(f'  Non-AI users   : n={len(non_grp):,}  mean={non_grp.mean():.2f}  median={non_grp.median():.1f}')
print(f'  Difference     : {ai_grp.mean()-non_grp.mean():.2f} pts  (AI users slightly lower)')

# t-test
t_stat, t_p = stats.ttest_ind(ai_grp, non_grp)
print(f'\n  t-test: t={t_stat:.3f}, p={t_p:.4f}')
print(f'  Statistical significance: {"Yes (p<0.05)" if t_p < 0.05 else "No (p>0.05)"}')
print()
print('  Interpretation: AI users are marginally less satisfied.')
print('  The utility-threat tension slightly erodes job satisfaction.')
print('  This is supporting evidence for Paradox 3, not the core proof.')

p3_jobsat_df = pd.DataFrame([
    {'Group':'Uses AI', 'n':len(ai_grp), 'Mean_JobSat':round(ai_grp.mean(),2), 'Median_JobSat':ai_grp.median()},
    {'Group':'Does Not Use AI', 'n':len(non_grp), 'Mean_JobSat':round(non_grp.mean(),2), 'Median_JobSat':non_grp.median()},
])
print('\n[P3 JobSat comparison saved for export]')

Job Satisfaction by AI Adoption Status:
  (Only employed respondents who answered JobSat: n=26,670)

  AI users       : n=20,140  mean=7.23  median=8.0
  Non-AI users   : n=4,778  mean=7.13  median=8.0
  Difference     : 0.10 pts  (AI users slightly lower)

  t-test: t=3.063, p=0.0022
  Statistical significance: Yes (p<0.05)

  Interpretation: AI users are marginally less satisfied.
  The utility-threat tension slightly erodes job satisfaction.
  This is supporting evidence for Paradox 3, not the core proof.

[P3 JobSat comparison saved for export]


---
# PART D — Create html dashboard

## Cell 16 — Build trust_paradox_powerbi.csv (37 columns for Power BI)

In [17]:
# ════════════════════════════════════════════════════════════════════
# CELL 16 — Build AI Trust Paradox Interactive HTML Dashboard
# Run after all cells 1-15 are complete
# Output: AI_Trust_Paradox_Dashboard.html (downloads automatically)
# ════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy  as np
import base64
import json   as _json
from IPython.display import Javascript, display

# ── Re-compute all summary variables ─────────────────────────────
trust_ans  = df[df['AIAcc_clean'] != 'No Response'].copy()
n_trust    = len(trust_ans)
uses_ans   = df[df['UsesAI'] != 'No Response']
adopt_rate = (uses_ans['UsesAI'] == 'Uses AI').mean() * 100
ht_pct     = (trust_ans['AIAcc_clean'] == 'Highly trust').mean() * 100
ad_pct     = trust_ans['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
mean_sc    = trust_ans['AIAcc_score'].mean()

TRUST_ORDER = ['Highly trust','Somewhat trust','Neither trust nor distrust',
               'Somewhat distrust','Highly distrust']
trust_dist = []
for level in TRUST_ORDER:
    cnt = int((trust_ans['AIAcc_clean'] == level).sum())
    pct = round(cnt / n_trust * 100, 1)
    trust_dist.append({'level': level, 'count': cnt, 'pct': pct})

BAND_ORDER = ['Early Career (0-2 yrs)','Mid-Early (3-5 yrs)',
               'Mid (6-10 yrs)','Experienced (10+ yrs)']
exp_rows = []
for band in BAND_ORDER:
    g  = trust_ans[trust_ans['ExperienceBand'] == band]
    ht = round((g['AIAcc_clean'] == 'Highly trust').mean() * 100, 1)
    ad = round(g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100, 1)
    exp_rows.append({'band': band, 'ht': ht, 'ad': ad})

country_data = trust_ans.groupby('Country').apply(
    lambda g: pd.Series({
        'n':  len(g),
        'ht': round((g['AIAcc_clean']=='Highly trust').mean()*100, 1),
        'ad': round(g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean()*100, 1),
    })
).reset_index()
country_data = country_data[country_data['n'] >= 200].sort_values('ht', ascending=False)

sent_ans   = df[df['AISent_clean'].isin(['Very favorable','Favorable','Indifferent',
                                          'Unfavorable','Very unfavorable'])].copy()
pos_sent   = round(sent_ans['AISent_clean'].isin(['Very favorable','Favorable']).mean()*100, 1)
pos_trust2 = round(trust_ans['AIAcc_clean'].isin(['Highly trust','Somewhat trust']).mean()*100, 1)
ethics_gap = round(pos_sent - pos_trust2, 1)
mean_sent  = round(df['AISent_score'].mean(), 3)
mean_trust = round(trust_ans['AIAcc_score'].mean(), 3)

TOOL_ORDER = ['1 Tool','2-3 Tools','4+ Tools']
tool_trust = df[(df['ToolGroup'] != 'No Response') & (df['AIAcc_clean'] != 'No Response')].copy()
tool_rows  = []
for grp in TOOL_ORDER:
    g  = tool_trust[tool_trust['ToolGroup'] == grp]
    ad = round(g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean()*100, 1)
    tool_rows.append({'grp': grp, 'ad': ad, 'n': len(g)})

frust_cols = {
    'Frust_AlmostRight'     : 'Almost right, needs changes',
    'Frust_Debugging'       : 'Debugging more time-consuming',
    'Frust_LessConfident'   : 'Less confident in own skills',
    'Frust_HardToUnderstand': 'Hard to understand why it worked',
}
frust_rows = []
for col, label in frust_cols.items():
    if col in df.columns:
        cnt = int(df[col].sum())
        pct = round(cnt / len(df) * 100, 1)
        frust_rows.append({'label': label, 'count': cnt, 'pct': pct})

threat_ans  = df[df['AIThreat_clean'] != 'No Response'].copy()
n_threat    = len(threat_ans)
threat_dist = {}
for val in ['No', "I'm not sure", 'Yes']:
    cnt = int((threat_ans['AIThreat_clean'] == val).sum())
    pct = round(cnt / n_threat * 100, 1)
    threat_dist[val] = {'count': cnt, 'pct': pct}

trend_not  = [70.0, 68.0, round(threat_dist['No']['pct'], 1)]
trend_sure = [18.0, 19.0, round(threat_dist["I'm not sure"]['pct'], 1)]
trend_yes  = [12.0, 13.0, round(threat_dist['Yes']['pct'], 1)]

threat_exp_rows = []
for band in BAND_ORDER:
    g   = threat_ans[threat_ans['ExperienceBand'] == band]
    thr = round((g['AIThreat_clean'] == 'Yes').mean() * 100, 1)
    ht_ref = {'Early Career (0-2 yrs)': 10.0, 'Mid-Early (3-5 yrs)': 5.4,
               'Mid (6-10 yrs)': 2.8, 'Experienced (10+ yrs)': 2.3}.get(band, 0)
    threat_exp_rows.append({'band': band, 'thr': thr, 'ht': ht_ref})

nr_ans  = df[df['NewRole'].notna()].copy()
nr_rows = []
for val, cnt in nr_ans['NewRole'].value_counts().items():
    pct = round(cnt / len(nr_ans) * 100, 1)
    nr_rows.append({'label': str(val)[:55], 'count': int(cnt), 'pct': pct})

print("Variables computed:")
print(f"  Adoption rate : {adopt_rate:.1f}%")
print(f"  High trust    : {ht_pct:.1f}%")
print(f"  Any distrust  : {ad_pct:.1f}%")
print(f"  Ethics gap    : {ethics_gap:.1f} pp")
print(f"  Not Threatened: {threat_dist['No']['pct']:.1f}%")
print()
print("Building dashboard HTML...")

# ── Convert to JSON for JavaScript ───────────────────────────────
trust_dist_js    = _json.dumps(trust_dist)
exp_rows_js      = _json.dumps(exp_rows)
country_js       = _json.dumps(country_data[['Country','ht','ad']].head(12).to_dict('records'))
tool_rows_js     = _json.dumps(tool_rows)
frust_rows_js    = _json.dumps(sorted(frust_rows, key=lambda x: -x['count']))
threat_dist_js   = _json.dumps([
    {'label': 'Not Threatened', 'pct': threat_dist['No']['pct'],          'color': '#1A7A4A'},
    {'label': 'Uncertain',      'pct': threat_dist["I'm not sure"]['pct'],'color': '#D97706'},
    {'label': 'Threatened',     'pct': threat_dist['Yes']['pct'],          'color': '#9B1C1C'},
])
trend_js         = _json.dumps({'years': [2023,2024,2025],
                                 'not': trend_not, 'sure': trend_sure, 'yes': trend_yes})
threat_exp_js    = _json.dumps(threat_exp_rows)
nr_rows_js       = _json.dumps(nr_rows[:5])

# ── Build HTML ────────────────────────────────────────────────────
adopt_str      = f"{adopt_rate:.1f}"
ht_str         = f"{ht_pct:.1f}"
ad_str         = f"{ad_pct:.1f}"
mean_sc_str    = f"{mean_sc:.3f}"
nt_str         = f"{threat_dist['No']['pct']:.1f}"
nu_str         = f"{threat_dist[chr(73)+chr(39)+'m not sure']['pct']:.1f}"
nt_yes_str     = f"{threat_dist['Yes']['pct']:.1f}"
ps_str         = f"{pos_sent:.1f}"
pt_str         = f"{pos_trust2:.1f}"
eg_str         = f"{ethics_gap:.1f}"
ms_str         = f"{mean_sent:.3f}"
mt_str         = f"{mean_trust:.3f}"
sev_str        = f"{adopt_rate - ht_pct:.1f}"
tno_str        = str(trend_not[-1])

html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>AI Trust Paradox Dashboard — AICW 2026</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Segoe UI',Calibri,sans-serif;background:#F0F2F6;color:#1A202C}
.strip{background:#1B3A6B;color:#fff;padding:10px 24px 8px;font-size:13px;border-bottom:3px solid #0D7377}
.strip b{font-size:15px}
.tabbar{background:#1B3A6B;display:flex;padding:0 24px;gap:4px;align-items:flex-end}
.tab{padding:10px 20px;cursor:pointer;color:#aac4f0;font-size:13px;font-weight:600;
     border-radius:6px 6px 0 0;border:none;background:transparent;transition:.2s}
.tab:hover,.tab.active{background:#F0F2F6;color:#1B3A6B}
.page{display:none;padding:20px 24px}
.page.active{display:block}
.kpi-row{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:16px 0}
.kpi{background:#fff;border-radius:8px;padding:16px;box-shadow:0 1px 4px rgba(0,0,0,.1);border-top:4px solid #0D7377}
.kpi .val{font-size:30px;font-weight:700}
.kpi .lbl{font-size:11px;color:#9CA3AF;margin-top:4px;text-transform:uppercase;letter-spacing:.5px}
.kpi .sub{font-size:11px;color:#9CA3AF;margin-top:2px;font-style:italic}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:16px 0}
.grid3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px;margin:16px 0}
.card{background:#fff;border-radius:8px;padding:16px;box-shadow:0 1px 4px rgba(0,0,0,.1)}
.card h3{font-size:13px;color:#1B3A6B;margin-bottom:12px;font-weight:700;
         border-bottom:2px solid #F0F2F6;padding-bottom:6px}
canvas{max-height:280px}
.box{border-left:5px solid #0D7377;background:#f0f4fb;padding:12px 16px;
     border-radius:0 6px 6px 0;margin:10px 0;font-size:13px;line-height:1.7}
.box.red{border-color:#9B1C1C;background:#FEE2E2}
.box.amber{border-color:#92400E;background:#FFFBEB}
.box.purple{border-color:#4C1D95;background:#EDE9FE}
.stat-tag{display:inline-block;background:#1B3A6B;color:#fff;
          font-size:11px;font-weight:700;padding:2px 8px;border-radius:12px;margin:2px}
footer{background:#1B3A6B;color:#aac4f0;padding:10px 24px;font-size:12px;text-align:center;margin-top:24px}
</style>
</head>
<body>

<div class="strip">
  <b>AI Trust Paradox Dashboard</b> &nbsp;·&nbsp;
  AICW Fellowship 2026 &nbsp;·&nbsp; Galgotias University &nbsp;·&nbsp;
  Dr. Chandni Rani &nbsp;·&nbsp; Guide: Ms. R Uma Maheswari &nbsp;·&nbsp;
  Stack Overflow Developer Survey 2025 &nbsp;·&nbsp; N = 48,195 &nbsp;·&nbsp; 177 countries
</div>

<div class="tabbar">
  <button class="tab active" onclick="showPage(0)">📊 Overview</button>
  <button class="tab"        onclick="showPage(1)">🔴 Paradox 1 — Use-Despite-Distrust</button>
  <button class="tab"        onclick="showPage(2)">🟡 Paradox 2 — Ethics Gap</button>
  <button class="tab"        onclick="showPage(3)">🟣 Paradox 3 — Utility-Threat</button>
</div>

<!-- PAGE 0 — OVERVIEW -->
<div class="page active" id="p0">
  <div class="kpi-row">
    <div class="kpi">
      <div class="val" style="color:#1A7A4A">""" + adopt_str + """%</div>
      <div class="lbl">AI Adoption Rate</div>
      <div class="sub">↑ Highest ever recorded</div>
    </div>
    <div class="kpi">
      <div class="val" style="color:#9B1C1C">""" + ht_str + """%</div>
      <div class="lbl">High Trust Rate</div>
      <div class="sub">Only 1 in 32 developers</div>
    </div>
    <div class="kpi">
      <div class="val" style="color:#9B1C1C">""" + ad_str + """%</div>
      <div class="lbl">Any Distrust Rate</div>
      <div class="sub">Nearly 1 in 2 active users</div>
    </div>
    <div class="kpi">
      <div class="val" style="color:#92400E">""" + nt_str + """%</div>
      <div class="lbl">Not Threatened by AI</div>
      <div class="sub">↓ from 70% in 2023</div>
    </div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>Trust Level Distribution &nbsp;<small style="color:#9CA3AF">(n=32,717 answered)</small></h3>
      <canvas id="cOverviewDonut"></canvas>
      <p style="text-align:center;font-size:22px;font-weight:700;color:#9B1C1C;margin-top:8px">
        Mean Trust Score: """ + mean_sc_str + """ / 5.0
      </p>
      <p style="text-align:center;font-size:12px;color:#9CA3AF">below 3.0 neutral midpoint → leans distrust</p>
    </div>
    <div class="card">
      <h3>Three Paradoxes — Summary</h3>
      <div class="box red">
        <b>P1 — Use-Despite-Distrust</b><br>
        """ + adopt_str + """% use AI daily · only """ + ht_str + """% highly trust output<br>
        Paradox Severity: """ + sev_str + """ pts
        <br><span class="stat-tag">χ²=5,728.86</span>
        <span class="stat-tag">p&lt;0.0001</span>
        <span class="stat-tag">V=0.42 LARGE</span>
      </div>
      <div class="box amber">
        <b>P2 — Adoption-Ethics Gap</b><br>
        Sentiment """ + ps_str + """% vs Trust """ + pt_str + """% = """ + eg_str + """ pp gap<br>
        Score gap: """ + ms_str + """ (sentiment) vs """ + mt_str + """ (trust)
      </div>
      <div class="box purple">
        <b>P3 — Utility-Threat Tension</b><br>
        Not Threatened: 70% → 68% → """ + tno_str + """%<br>
        43.1% career impact · """ + nt_yes_str + """% actively threatened
        <br><span class="stat-tag">χ²=439.93</span>
        <span class="stat-tag">p&lt;0.0001</span>
      </div>
    </div>
  </div>
  <div class="grid3">
    <div class="card" style="text-align:center">
      <h3>Paradox Severity Score</h3>
      <div style="font-size:42px;font-weight:700;color:#9B1C1C;padding:16px 0">""" + sev_str + """</div>
      <div style="font-size:12px;color:#9CA3AF">Adoption% minus High Trust%</div>
    </div>
    <div class="card" style="text-align:center">
      <h3>Ethics Gap</h3>
      <div style="font-size:42px;font-weight:700;color:#92400E;padding:16px 0">""" + eg_str + """ pp</div>
      <div style="font-size:12px;color:#9CA3AF">Sentiment vs Analytical Trust</div>
    </div>
    <div class="card" style="text-align:center">
      <h3>Total Respondents</h3>
      <div style="font-size:42px;font-weight:700;color:#1B3A6B;padding:16px 0">48,195</div>
      <div style="font-size:12px;color:#9CA3AF">177 countries · SO Survey 2025</div>
    </div>
  </div>
</div>

<!-- PAGE 1 — PARADOX 1 -->
<div class="page" id="p1">
  <div class="kpi-row">
    <div class="kpi"><div class="val" style="color:#1A7A4A">""" + adopt_str + """%</div><div class="lbl">Adoption Rate</div></div>
    <div class="kpi"><div class="val" style="color:#9B1C1C">""" + ht_str + """%</div><div class="lbl">Highly Trust Output</div></div>
    <div class="kpi"><div class="val" style="color:#9B1C1C">""" + ad_str + """%</div><div class="lbl">Actively Distrust</div></div>
    <div class="kpi"><div class="val" style="color:#9B1C1C">""" + mean_sc_str + """</div><div class="lbl">Mean Trust Score / 5.0</div><div class="sub">Below 3.0 neutral</div></div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>Trust Distribution — All Respondents who Answered</h3>
      <canvas id="cTrustBar"></canvas>
    </div>
    <div class="card">
      <h3>Expertise Effect — Trust FALLS as Experience RISES</h3>
      <canvas id="cExpertise"></canvas>
    </div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>High Trust % by Country (top 12, min 200 respondents)</h3>
      <canvas id="cCountry"></canvas>
    </div>
    <div class="card">
      <h3>Key Finding — Paradox 1</h3>
      <div class="box red">
        <b>78.5% use AI daily — only 3.1% highly trust the output.</b><br><br>
        Among active AI users: <b>37.4% actively distrust</b> what they deploy every day.<br><br>
        Trust falls monotonically as experience rises:<br>
        Early Career 10.0% → Experienced 2.3%<br><br>
        The Technology Acceptance Model (TAM, Davis 1989) is <b>empirically FALSIFIED</b> for AI tools.
        <br><br>
        <span class="stat-tag">χ²=5,728.86</span>
        <span class="stat-tag">p&lt;0.0001</span>
        <span class="stat-tag">Cramér's V=0.42 (LARGE)</span>
      </div>
    </div>
  </div>
</div>

<!-- PAGE 2 — PARADOX 2 -->
<div class="page" id="p2">
  <div class="kpi-row">
    <div class="kpi"><div class="val" style="color:#1A7A4A">""" + ps_str + """%</div><div class="lbl">Positive Sentiment</div></div>
    <div class="kpi"><div class="val" style="color:#9B1C1C">""" + pt_str + """%</div><div class="lbl">Positive Trust</div></div>
    <div class="kpi"><div class="val" style="color:#92400E">""" + eg_str + """ pp</div><div class="lbl">Ethics Gap</div></div>
    <div class="kpi"><div class="val" style="color:#92400E">""" + ms_str + """ vs """ + mt_str + """</div><div class="lbl">Score Gap (Sent vs Trust)</div></div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>Ethics Gap — Sentiment vs Trust (same 1–5 scale)</h3>
      <canvas id="cEthicsGap"></canvas>
    </div>
    <div class="card">
      <h3>More Tools = MORE Distrust (Exposure Discovery Effect)</h3>
      <canvas id="cToolGroup"></canvas>
    </div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>Top AI Frustrations — Why Trust Stays Low</h3>
      <canvas id="cFrustration"></canvas>
    </div>
    <div class="card">
      <h3>Key Finding — Paradox 2</h3>
      <div class="box amber">
        <b>""" + eg_str + """ pp gap between positive sentiment (""" + ps_str + """%) and positive trust (""" + pt_str + """%).</b><br><br>
        Developers <i>enjoy</i> using AI tools — but analytically <i>distrust</i> the output.<br><br>
        Score gap: Sentiment mean <b>""" + ms_str + """</b> vs Trust mean <b>""" + mt_str + """</b> on same 1–5 scale.<br><br>
        More tool exposure compounds distrust: 1 tool 41.4% → 4+ tools 47.5%.<br><br>
        Top frustration: <b>"Almost right but needs changes every time"</b> — 42.5% of all developers.
      </div>
    </div>
  </div>
</div>

<!-- PAGE 3 — PARADOX 3 -->
<div class="page" id="p3">
  <div class="kpi-row">
    <div class="kpi"><div class="val" style="color:#1A7A4A">""" + nt_str + """%</div><div class="lbl">Not Threatened</div><div class="sub">↓ from 70% (2023)</div></div>
    <div class="kpi"><div class="val" style="color:#D97706">""" + nu_str + """%</div><div class="lbl">Uncertain</div></div>
    <div class="kpi"><div class="val" style="color:#9B1C1C">""" + nt_yes_str + """%</div><div class="lbl">Feel Threatened</div></div>
    <div class="kpi"><div class="val" style="color:#4C1D95">43.1%</div><div class="lbl">Career Impact</div><div class="sub">Considered / made change</div></div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>3-Year Trend — Job Threat Perception (Confirmed Decline)</h3>
      <canvas id="cTrend"></canvas>
    </div>
    <div class="card">
      <h3>Job Threat Distribution 2025</h3>
      <canvas id="cThreatDonut"></canvas>
    </div>
  </div>
  <div class="grid2">
    <div class="card">
      <h3>Cross-Paradox — Threatened % vs High Trust % by Experience</h3>
      <canvas id="cThreatExp"></canvas>
    </div>
    <div class="card">
      <h3>Career Change Due to AI — Spectrum</h3>
      <canvas id="cCareer"></canvas>
    </div>
  </div>
  <div class="box purple">
    <b>Key Finding — Paradox 3:</b> Not Threatened declined 3 consecutive years:
    70% → 68% → """ + tno_str + """% (χ²=439.93, p&lt;0.0001).
    Cross-paradox insight: Experienced developers trust AI LEAST (P1)
    but feel LEAST threatened (P3) — expertise drives both scepticism AND professional security.
  </div>
</div>

<footer>
  AI Trust Paradox Dashboard &nbsp;·&nbsp; AICW Fellowship 2026 &nbsp;·&nbsp;
  Edunet Foundation (Microsoft · LinkedIn · SAP) &nbsp;·&nbsp;
  Stack Overflow Developer Survey 2025 &nbsp;·&nbsp;
  Dr. Chandni Rani &nbsp;·&nbsp; Guide: Ms. R Uma Maheswari
</footer>

<script>
const trustDist   = """ + trust_dist_js   + """;
const expRows     = """ + exp_rows_js     + """;
const countryData = """ + country_js      + """;
const toolRows    = """ + tool_rows_js    + """;
const frustRows   = """ + frust_rows_js   + """;
const threatDist  = """ + threat_dist_js  + """;
const trend       = """ + trend_js        + """;
const threatExp   = """ + threat_exp_js   + """;
const nrRows      = """ + nr_rows_js      + """;

function showPage(idx) {
  document.querySelectorAll('.page').forEach((p,i) => p.classList.toggle('active', i===idx));
  document.querySelectorAll('.tab').forEach((t,i)  => t.classList.toggle('active', i===idx));
}

Chart.defaults.font.family = "'Segoe UI', Calibri, sans-serif";
Chart.defaults.font.size   = 12;

const TC = ['#1A7A4A','#52BE80','#9CA3AF','#E74C3C','#9B1C1C'];
const BC = ['#1A7A4A','#52BE80','#D97706','#9B1C1C'];

new Chart(document.getElementById('cOverviewDonut'), {
  type: 'doughnut',
  data: { labels: trustDist.map(d=>d.level+' ('+d.pct+'%)'),
    datasets: [{ data: trustDist.map(d=>d.pct), backgroundColor: TC,
                 borderWidth: 3, borderColor: '#fff' }] },
  options: { cutout: '55%', plugins: { legend: { position: 'bottom', labels: { font: { size: 11 } } } } }
});

new Chart(document.getElementById('cTrustBar'), {
  type: 'bar',
  data: { labels: trustDist.map(d=>d.level),
    datasets: [{ label:'% of Respondents', data: trustDist.map(d=>d.pct),
                 backgroundColor: TC, borderRadius: 4 }] },
  options: { indexAxis: 'y', plugins: { legend: { display: false } }, scales: { x: { max: 40 } } }
});

new Chart(document.getElementById('cExpertise'), {
  type: 'bar',
  data: { labels: expRows.map(r=>r.band.replace(' yrs)',')')),
    datasets: [
      { label: 'High Trust %',   data: expRows.map(r=>r.ht), backgroundColor: '#1A7A4A', borderRadius: 4 },
      { label: 'Any Distrust %', data: expRows.map(r=>r.ad), backgroundColor: '#E74C3C', borderRadius: 4 }
    ] },
  options: { plugins: { legend: { position: 'bottom' } }, scales: { y: { beginAtZero: true } } }
});

new Chart(document.getElementById('cCountry'), {
  type: 'bar',
  data: { labels: countryData.map(c=>c.Country),
    datasets: [{ label: 'High Trust %', data: countryData.map(c=>c.ht),
      backgroundColor: countryData.map(c=>c.ht<3?'#9B1C1C':c.ht<6?'#D97706':'#1A7A4A'),
      borderRadius: 4 }] },
  options: { indexAxis: 'y', plugins: { legend: { display: false } }, scales: { x: { beginAtZero: true } } }
});

new Chart(document.getElementById('cEthicsGap'), {
  type: 'bar',
  data: { labels: ['Positive Sentiment','Positive Trust (Highly + Somewhat)'],
    datasets: [{ label: '%', data: [""" + ps_str + """, """ + pt_str + """],
                 backgroundColor: ['#1A7A4A','#9B1C1C'], borderRadius: 6 }] },
  options: { indexAxis: 'y', plugins: { legend: { display: false } },
    scales: { x: { max: 75, beginAtZero: true } } }
});

new Chart(document.getElementById('cToolGroup'), {
  type: 'bar',
  data: { labels: toolRows.map(r=>r.grp+' (n='+r.n.toLocaleString()+')'),
    datasets: [{ label: 'Any Distrust %', data: toolRows.map(r=>r.ad),
      backgroundColor: ['#FEF3C7','#FFFBEB','#9B1C1C'],
      borderColor: ['#D97706','#D97706','#9B1C1C'], borderWidth: 2, borderRadius: 6 }] },
  options: { plugins: { legend: { display: false } }, scales: { y: { min: 35, max: 52 } } }
});

new Chart(document.getElementById('cFrustration'), {
  type: 'bar',
  data: { labels: frustRows.map(r=>r.label),
    datasets: [{ label: 'Count', data: frustRows.map(r=>r.count),
                 backgroundColor: ['#9B1C1C','#9B1C1C','#D97706','#D97706'], borderRadius: 4 }] },
  options: { indexAxis: 'y', plugins: { legend: { display: false } }, scales: { x: { beginAtZero: true } } }
});

new Chart(document.getElementById('cTrend'), {
  type: 'line',
  data: { labels: trend.years,
    datasets: [
      { label:'Not Threatened', data:trend.not,  borderColor:'#1A7A4A',
        backgroundColor:'rgba(26,122,74,.1)', tension:.3, fill:true, pointRadius:6, pointBackgroundColor:'#1A7A4A' },
      { label:'Uncertain',      data:trend.sure, borderColor:'#D97706',
        tension:.3, fill:false, pointRadius:6 },
      { label:'Threatened',     data:trend.yes,  borderColor:'#9B1C1C',
        backgroundColor:'rgba(155,28,28,.08)', tension:.3, fill:true, pointRadius:6, pointBackgroundColor:'#9B1C1C' }
    ] },
  options: { plugins: { legend: { position: 'bottom' } },
    scales: { y: { min:5, max:80, ticks: { callback: v=>v+'%' } } } }
});

new Chart(document.getElementById('cThreatDonut'), {
  type: 'doughnut',
  data: { labels: threatDist.map(d=>d.label+' ('+d.pct+'%)'),
    datasets: [{ data: threatDist.map(d=>d.pct),
                 backgroundColor: threatDist.map(d=>d.color), borderWidth:3, borderColor:'#fff' }] },
  options: { cutout: '55%', plugins: { legend: { position: 'bottom' } } }
});

new Chart(document.getElementById('cThreatExp'), {
  type: 'bar',
  data: { labels: threatExp.map(r=>r.band.replace(' yrs)',')')),
    datasets: [
      { label: 'Threatened %',   data: threatExp.map(r=>r.thr), backgroundColor: '#9B1C1C', borderRadius: 4 },
      { label: 'P1 High Trust %',data: threatExp.map(r=>r.ht),  backgroundColor: '#1A7A4A', borderRadius: 4 }
    ] },
  options: { plugins: { legend: { position: 'bottom' } }, scales: { y: { beginAtZero: true } } }
});

new Chart(document.getElementById('cCareer'), {
  type: 'bar',
  data: { labels: nrRows.map(r=>r.label.substring(0,38)+'...'),
    datasets: [{ label: '%', data: nrRows.map(r=>r.pct),
      backgroundColor: ['#9CA3AF','#A78BFA','#7C3AED','#4C1D95','#9B1C1C'], borderRadius: 4 }] },
  options: { indexAxis: 'y', plugins: { legend: { display: false } }, scales: { x: { beginAtZero: true } } }
});
</script>
</body>
</html>"""

# ── Save and download ─────────────────────────────────────────────
with open('AI_Trust_Paradox_Dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f"Saved: AI_Trust_Paradox_Dashboard.html  ({len(html)//1024} KB)")
print()

b64 = base64.b64encode(html.encode('utf-8')).decode('utf-8')
js  = f"""
var bytes = Uint8Array.from(atob('{b64}'), c => c.charCodeAt(0));
var blob  = new Blob([bytes], {{type:'text/html'}});
var url   = URL.createObjectURL(blob);
var a     = document.createElement('a');
a.href    = url;  a.download = 'AI_Trust_Paradox_Dashboard.html';
document.body.appendChild(a);  a.click();
document.body.removeChild(a);  URL.revokeObjectURL(url);
"""
display(Javascript(js))

print("Download triggered — check your Downloads folder.")
print()
print("OBJECTIVE 2 COMPLETE")
print("Open AI_Trust_Paradox_Dashboard.html in Chrome or any browser.")
print("4 pages: Overview · Paradox 1 · Paradox 2 · Paradox 3")

Variables computed:
  Adoption rate : 78.5%
  High trust    : 3.1%
  Any distrust  : 45.9%
  Ethics gap    : 28.4 pp
  Not Threatened: 63.8%

Building dashboard HTML...
Saved: AI_Trust_Paradox_Dashboard.html  (18 KB)



<IPython.core.display.Javascript object>

Download triggered — check your Downloads folder.

OBJECTIVE 2 COMPLETE
Open AI_Trust_Paradox_Dashboard.html in Chrome or any browser.
4 pages: Overview · Paradox 1 · Paradox 2 · Paradox 3


## Cell 17 — Save all 4 analysis summary CSVs